# `5.compile` - Composing the MLIR compiler stack

This notebook mirrors the purpose of `examples/5.compile.ipynb`: start with a Toy program that crosses the quantum/classical boundary, then inspect and execute it through the available ISA and QEC transformations. The selector interface remains stable while quantum dataflow changes underneath it.

In [ ]:
%load_ext qstack_mlir.jupyter

import logging

logger = logging.getLogger("qstack")
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
    logger.addHandler(handler)

## 1. Source program

The Toy program retries until an ancilla measurement is zero. This gives every compiler stage both quantum operations and a host selector contract to preserve.

In [ ]:
%%qasm toy_program
QSTACKQASM 0.1;
include "qstack/toy.inc";

extern selector repeat_until_zero(bit) -> int;

def prepare_zero(qubit q) {
  qreg ancilla[1];
  bit m;
  mix q;
  entangle q, ancilla[0];
  measure ancilla[0] -> m;
  switch (repeat_until_zero(m)) {
    case 0: { }
    case 1: { prepare_zero q; }
  }
}

qreg q[1];
creg c[1];
prepare_zero q[0];
measure q[0] -> c[0];

In [ ]:
from qstack_mlir.runtime import CallbackRegistry, Machine

registry = CallbackRegistry()

@registry.selector("repeat_until_zero")
def _repeat_until_zero(*, b0):
    return "0" if b0 == 0 else "1"

print(toy_program)
Machine(toy_program, num_qubits=4, registry=registry).shots("main", 200).plot_histogram()

## 2. Toy to Cliffords

`compile_toy_to_cliffords` rewrites `mix` to `h` and `entangle` to `cx` in place. The selector declaration, `qstack.select`, continuation functions, and recursive call remain visible and unchanged.

In [ ]:
from qstack_mlir.passes.toy2cliffords import compile_toy_to_cliffords

clifford_program = toy_program.clone()
compile_toy_to_cliffords(clifford_program)
print(clifford_program)
Machine(clifford_program, num_qubits=4, registry=registry).shots("main", 200).plot_histogram()

## 3. Cliffords to H2

The hardware lowering expands Cliffords into parameterized `h2.u1`, `h2.rz`, and `h2.zz` operations. Cloning keeps the Clifford module available for the independent QEC branches below.

In [ ]:
from qstack_mlir.passes.cliffords2h2 import compile_cliffords_to_h2

h2_program = clifford_program.clone()
compile_cliffords_to_h2(h2_program)
print(h2_program)
Machine(h2_program, num_qubits=4, registry=registry).shots("main", 200).plot_histogram()

## 4. Repetition-3 encoding

Rep3 widens every logical qubit to three physical qubits and inserts `qstack.decode @majority_vote` after physical measurements. The original selector still consumes one logical bit.

In [ ]:
from qstack_mlir.passes.rep3_trivial import compile_rep3, register_rep3_callbacks

rep3_program = compile_rep3(clifford_program)
register_rep3_callbacks(registry)
print(rep3_program)
Machine(rep3_program, num_qubits=8, registry=registry).shots("main", 200).plot_histogram()

## 5. Rep3 followed by H2

ISA lowering composes after QEC: the three physical copies are retained while every generated Clifford is lowered to H2-native operations.

In [ ]:
rep3_h2_program = rep3_program.clone()
compile_cliffords_to_h2(rep3_h2_program)
print(rep3_h2_program)
Machine(rep3_h2_program, num_qubits=8, registry=registry).shots("main", 200).plot_histogram()

## 6. Concatenated repetition code

Applying Rep3 again produces nine physical qubits per logical qubit and nested majority-vote decoding. The callback registry does not change.

In [ ]:
rep9_program = compile_rep3(rep3_program)
print(rep9_program)

## 7. Steane encoding

Steane widens each logical qubit to seven physical qubits and adds encoded-zero preparation, syndrome extraction, static correction menus, and logical decoding. Printing the module is the point here: it exposes the much larger structural transformation.

In [ ]:
from qstack_mlir.passes.steane import compile_steane, register_steane_callbacks

steane_program = compile_steane(clifford_program)
register_steane_callbacks(registry)
print(steane_program)

## 8. Trace an encoded evaluation

A single Rep3 shot keeps the trace readable while showing physical gates, measurements, majority decoding, and selector choices through the built-in `qstack` logger.

In [ ]:
logger.setLevel(logging.DEBUG)
trace = Machine(rep3_program, num_qubits=8, registry=registry).shots("main", 1).data
logger.setLevel(logging.INFO)
trace